# Notebook 08 — Physical Scaling and Secular Frequency Estimates

This notebook extends the reduced ion-trap workflow by adding simple physical scaling for a trapped ion.

It focuses on:

- solving Laplace's equation for a surface-style electrode layout
- computing electric fields and a normalized RF-style pseudopotential
- locating a candidate trap point and extracting local curvature
- introducing a simple physical length scale and ion parameters
- converting curvature-based diagnostics into approximate secular-frequency estimates

This remains a reduced model, but it starts to connect the notebook workflow to hardware-facing trapped-ion quantities.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Physical note

A common pseudopotential approximation for an RF ion trap is:

$$
\Phi_{\mathrm{pseudo}} = \frac{q^2 |E|^2}{4 m \Omega^2}
$$

where:

- $q$ is the ion charge
- $m$ is the ion mass
- $\Omega$ is the RF angular drive frequency
- $E$ is the electric-field magnitude

Near a local minimum, the Hessian of the pseudopotential determines local harmonic curvature. If the local pseudopotential is approximately quadratic,

$$
\Phi \approx \Phi_0 + \frac{1}{2} m \omega_x^2 x^2 + \frac{1}{2} m \omega_y^2 y^2
$$

then curvature can be converted into approximate secular frequencies.

In this notebook we introduce a simple physical length scale so the reduced model can produce order-of-magnitude frequency estimates.


## Grid, geometry, and fixed voltages


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Example geometry and voltages based on earlier notebooks
center_width = 0.6
gap = 0.25
side_width = 1.3
v_rf = 0.8
v_dc = -2.0

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')
print(f'center_width = {center_width:.3f}, gap = {gap:.3f}, side_width = {side_width:.3f}')
print(f'v_rf = {v_rf:.3f}, v_dc = {v_dc:.3f}')


## Physical scaling choices


In [ ]:
# Example ion parameters: 40Ca+
e_charge = 1.602176634e-19          # Coulomb
amu = 1.66053906660e-27             # kg
ion_mass = 40.0 * amu               # kg

# Example RF drive
rf_frequency_hz = 20e6              # 20 MHz
Omega = 2.0 * np.pi * rf_frequency_hz

# Physical length scale for one notebook coordinate unit
# Here we choose 1 notebook unit = 100 micrometers
length_scale_m = 100e-6

print('Physical scaling assumptions:')
print(f'Ion charge q = {e_charge:.3e} C')
print(f'Ion mass m = {ion_mass:.3e} kg')
print(f'RF drive frequency = {rf_frequency_hz/1e6:.1f} MHz')
print(f'Omega = {Omega:.3e} rad/s')
print(f'Length scale = {length_scale_m:.3e} m per notebook length unit')


## Helper functions


In [ ]:
def idx(i, j, nx):
    return i * nx + j

def make_electrode_masks(center_width, gap, side_width):
    center_half = center_width / 2.0
    left_center = -(center_half + gap + side_width / 2.0)
    right_center = +(center_half + gap + side_width / 2.0)

    left_mask = (x >= left_center - side_width / 2.0) & (x <= left_center + side_width / 2.0)
    center_mask = (x >= -center_half) & (x <= center_half)
    right_mask = (x >= right_center - side_width / 2.0) & (x <= right_center + side_width / 2.0)
    remaining_ground = ~(left_mask | center_mask | right_mask)
    return left_mask, center_mask, right_mask, remaining_ground

def build_boundary_problem(left_mask, center_mask, right_mask, remaining_ground, v_rf, v_dc):
    V = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros((ny, nx), dtype=bool)

    V[:, 0] = 0.0
    V[:, -1] = 0.0
    V[-1, :] = 0.0
    fixed[:, 0] = True
    fixed[:, -1] = True
    fixed[-1, :] = True

    bottom = 0
    V[bottom, left_mask] = v_dc
    V[bottom, center_mask] = v_rf
    V[bottom, right_mask] = v_dc
    V[bottom, remaining_ground] = 0.0
    fixed[bottom, :] = True
    return V, fixed

def solve_laplace(V, fixed):
    N = nx * ny
    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)
    cx = 1.0 / dx**2
    cy = 1.0 / dy**2
    c0 = -2.0 * (cx + cy)

    for i in range(ny):
        for j in range(nx):
            k = idx(i, j, nx)
            if fixed[i, j]:
                A[k, k] = 1.0
                b[k] = V[i, j]
            else:
                A[k, idx(i, j, nx)] = c0
                A[k, idx(i, j - 1, nx)] = cx
                A[k, idx(i, j + 1, nx)] = cx
                A[k, idx(i - 1, j, nx)] = cy
                A[k, idx(i + 1, j, nx)] = cy

    sol = spsolve(A.tocsr(), b)
    return sol.reshape((ny, nx))

def compute_reduced_fields(Vsol):
    dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
    Ex_red = -dV_dx
    Ey_red = -dV_dy
    E_red_mag = np.sqrt(Ex_red**2 + Ey_red**2)
    return Ex_red, Ey_red, E_red_mag

def find_candidate_point(Phi):
    margin_i_low = 2
    margin_i_high = 3
    margin_j = 3
    i_slice = slice(margin_i_low, ny - margin_i_high)
    j_slice = slice(margin_j, nx - margin_j)
    Phi_search = Phi[i_slice, j_slice]
    min_flat = np.argmin(Phi_search)
    min_i_local, min_j_local = np.unravel_index(min_flat, Phi_search.shape)
    min_i = min_i_local + margin_i_low
    min_j = min_j_local + margin_j
    return min_i, min_j

def second_derivatives(F, i, j, dx_eff, dy_eff):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx_eff**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy_eff**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx_eff * dy_eff)
    return d2F_dx2, d2F_dy2, d2F_dxdy


## Build geometry and solve Laplace equation


In [ ]:
left_mask, center_mask, right_mask, remaining_ground = make_electrode_masks(
    center_width=center_width,
    gap=gap,
    side_width=side_width,
)

V, fixed = build_boundary_problem(
    left_mask=left_mask,
    center_mask=center_mask,
    right_mask=right_mask,
    remaining_ground=remaining_ground,
    v_rf=v_rf,
    v_dc=v_dc,
)

Vsol = solve_laplace(V, fixed)
print(f'Potential range: [{Vsol.min():.3f}, {Vsol.max():.3f}]')


## Reduced electric field and physical pseudopotential

The notebook solver works in reduced coordinates. To assign physical units, we use:

$$
x_{\mathrm{phys}} = L\,x_{\mathrm{notebook}}
$$

So the physical electric field scales as:

$$
E_{\mathrm{phys}} = \frac{1}{L} E_{\mathrm{reduced}}
$$

because the gradient introduces one inverse length factor.


In [ ]:
Ex_red, Ey_red, E_red_mag = compute_reduced_fields(Vsol)

# Convert reduced field to physical units (V/m) using the length scale
Ex_phys = Ex_red / length_scale_m
Ey_phys = Ey_red / length_scale_m
E_phys_mag = np.sqrt(Ex_phys**2 + Ey_phys**2)

# Physical pseudopotential in Joules
Phi_phys_J = (e_charge**2 * E_phys_mag**2) / (4.0 * ion_mass * Omega**2)

# Convert to electron-volts for readability
Phi_phys_eV = Phi_phys_J / e_charge

print(f'Physical field magnitude range: [{E_phys_mag.min():.3e}, {E_phys_mag.max():.3e}] V/m')
print(f'Pseudopotential range: [{Phi_phys_eV.min():.3e}, {Phi_phys_eV.max():.3e}] eV')


## Locate candidate trap point


In [ ]:
min_i, min_j = find_candidate_point(Phi_phys_J)
trap_x = x[min_j]
trap_y = y[min_i]
trap_x_m = trap_x * length_scale_m
trap_y_m = trap_y * length_scale_m
trap_value_eV = Phi_phys_eV[min_i, min_j]

center = Phi_phys_J[min_i, min_j]
neighbors = np.array([
    Phi_phys_J[min_i - 1, min_j],
    Phi_phys_J[min_i + 1, min_j],
    Phi_phys_J[min_i, min_j - 1],
    Phi_phys_J[min_i, min_j + 1],
])
is_local_min = bool(np.all(center <= neighbors))

print(f'Candidate trap point: x = {trap_x:.3f}, y = {trap_y:.3f} (notebook units)')
print(f'Candidate trap point: x = {trap_x_m*1e6:.1f} um, y = {trap_y_m*1e6:.1f} um')
print(f'Local-minimum check: {is_local_min}')
print(f'Pseudopotential at trap point: {trap_value_eV:.3e} eV')


## Local curvature and secular-frequency estimates

We now compute the Hessian of the **physical** pseudopotential in SI units. Near a quadratic minimum,

$$
\frac{\partial^2 \Phi}{\partial x^2} = m \omega_x^2
$$

and similarly for the principal axes. So if $\lambda_i$ are the Hessian eigenvalues in units of J/m²,

$$
\omega_i = \sqrt{\lambda_i / m}
$$

for positive-curvature directions.


In [ ]:
dx_phys = dx * length_scale_m
dy_phys = dy * length_scale_m

d2x, d2y, d2xy = second_derivatives(Phi_phys_J, min_i, min_j, dx_phys, dy_phys)
H_phys = np.array([
    [d2x, d2xy],
    [d2xy, d2y],
], dtype=float)

eigvals_phys, eigvecs_phys = np.linalg.eigh(H_phys)
valid_trap = bool(is_local_min and np.min(eigvals_phys) > 0.0)

omega_rad_s = np.sqrt(np.clip(eigvals_phys / ion_mass, 0.0, None))
omega_hz = omega_rad_s / (2.0 * np.pi)

print('Physical Hessian (J/m^2):')
print(H_phys)
print('\nPrincipal curvature eigenvalues (J/m^2):')
print(eigvals_phys)
print(f'\nValid trap by local curvature check: {valid_trap}')
print('Approximate secular frequencies:')
for k, f in enumerate(omega_hz, start=1):
    print(f'  mode {k}: {f/1e6:.3f} MHz')


## Visualize the physical pseudopotential landscape


In [ ]:
X, Y = np.meshgrid(x, y)
X_um = X * length_scale_m * 1e6
Y_um = Y * length_scale_m * 1e6

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.pcolormesh(X_um, Y_um, Phi_phys_eV, shading='auto')
cs = ax.contour(X_um, Y_um, Phi_phys_eV, levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([trap_x_m * 1e6], [trap_y_m * 1e6], s=90, label='Candidate trap point')
ax.plot(x[left_mask] * length_scale_m * 1e6, np.full(np.sum(left_mask), Y_um[0,0]), linewidth=7)
ax.plot(x[center_mask] * length_scale_m * 1e6, np.full(np.sum(center_mask), Y_um[0,0]), linewidth=7)
ax.plot(x[right_mask] * length_scale_m * 1e6, np.full(np.sum(right_mask), Y_um[0,0]), linewidth=7)
ax.set_title('Physical RF-Style Pseudopotential')
ax.set_xlabel('x (um)')
ax.set_ylabel('y (um)')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudopotential (eV)')
plt.tight_layout()
plt.show()


## Local cuts through the trap point


In [ ]:
j_window = slice(max(min_j - 8, 0), min(min_j + 9, nx))
i_window = slice(max(min_i - 8, 0), min(min_i + 9, ny))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x[j_window] * length_scale_m * 1e6, Phi_phys_eV[min_i, j_window])
ax.axvline(trap_x_m * 1e6, linestyle='--', linewidth=1)
ax.set_title('Local Pseudopotential Cut Along x')
ax.set_xlabel('x (um)')
ax.set_ylabel('Pseudopotential (eV)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y[i_window] * length_scale_m * 1e6, Phi_phys_eV[i_window, min_j])
ax.axvline(trap_y_m * 1e6, linestyle='--', linewidth=1)
ax.set_title('Local Pseudopotential Cut Along y')
ax.set_xlabel('y (um)')
ax.set_ylabel('Pseudopotential (eV)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Design summary


In [ ]:
print('Design Summary')
print('--------------')
print(f'Geometry: center_width = {center_width:.3f}, gap = {gap:.3f}, side_width = {side_width:.3f}')
print(f'Voltages: v_rf = {v_rf:.3f}, v_dc = {v_dc:.3f}')
print(f'Trap location: ({trap_x_m*1e6:.1f} um, {trap_y_m*1e6:.1f} um)')
print(f'Valid local trap: {valid_trap}')
print(f'Trap pseudopotential: {trap_value_eV:.3e} eV')
print('Approximate secular frequencies:')
for k, f in enumerate(omega_hz, start=1):
    print(f'  mode {k}: {f/1e6:.3f} MHz')


## Notes and next steps

This notebook introduces a simple physical scaling layer for the reduced trap-design workflow.

Important caveats:

- the notebook still uses a simplified 2D electrostatic model
- the pseudopotential uses a reduced boundary-voltage description rather than a full RF electrode simulation
- axial confinement and compensation fields are not yet included
- frequency estimates depend on the chosen physical length scale

Natural next steps:

- sweep physical RF frequency and length scale assumptions
- add explicit trap-depth diagnostics in eV
- compare multiple ion species
- include axial confinement and compensation fields
- connect the best geometry from Notebook 07 directly into this physical-scaling workflow

That progression will move the workflow closer to hardware-facing trapped-ion design estimates.
